# Checkpoint 2: Research Question Formation
## Twitter Sentiment140 — Defining Research Questions and Methodological Plan

**Author:** Rishitha  
**Dataset:** Twitter Sentiment140  
**Goal:** Define meaningful, feasible research questions supported by course techniques and at least one external technique, backed by additional EDA and initial method runs.

---
# Collaboration Declaration

## (1) Collaborators
None.

## (2) Web Sources
- Twitter Sentiment140 Dataset (Kaggle): https://www.kaggle.com/datasets/kazanova/sentiment140
- Scikit-learn Documentation: https://scikit-learn.org/stable/
- Hugging Face Transformers: https://huggingface.co/docs/transformers/index
- NLTK Documentation: https://www.nltk.org/
- BERTopic Documentation: https://maartengr.github.io/BERTopic/

## (3) AI Tools
Claude (Anthropic) was used to help structure the notebook, check code logic, and review explanations. All analytical decisions, interpretations, and research questions were developed independently.

## (4) Citations for Papers
- Go, A., Bhayani, R., & Huang, L. (2009). *Twitter Sentiment Classification using Distant Supervision.* Introduces the Sentiment140 dataset and labeling methodology.
- Barbieri, F., Camacho-Collados, J., Espinosa-Anke, L., & Neves, L. (2020). *TweetEval: Unified Benchmark and Comparative Evaluation for Tweet Classification.* Findings of EMNLP 2020. Motivates transformer-based models on Twitter text.
- Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.* NAACL 2019. Foundational paper for the beyond-course transformer technique.
- Blei, D. M., Ng, A. Y., & Jordan, M. I. (2003). *Latent Dirichlet Allocation.* JMLR. Foundational paper for LDA topic modeling used as a course technique.

---
# Section 1: Project Scope — Dataset and EDA Recap

## 1.1 Dataset Summary

**Dataset:** Twitter Sentiment140 (Kaggle)  
**Size:** ~1.6 million tweets  
**Format:** CSV with 6 fields — sentiment label (0=negative, 4=positive), tweet ID, date, query, username, tweet text  
**Target variable:** Binary sentiment label (negative vs. positive)

## 1.2 Key EDA Findings from Checkpoint 1

| Finding | Implication |
|---|---|
| **Balanced classes** (50% positive, 50% negative) | No class imbalance correction needed for classification |
| **~18K duplicate tweets** | Must deduplicate before modeling to avoid data leakage |
| **Short text** — most tweets under 30 words | TF-IDF produces very sparse features; context matters more than frequency |
| **TF-IDF sparsity ≈ 0.9999** on 50K features | Traditional bag-of-words may miss semantic nuance |
| **Top words** (just, good, day, love, work) are cross-sentiment | Discriminative signal requires richer representations |
| **Informal language** (lol, im, quot, amp) | Preprocessing and domain-adapted models are important |

## 1.3 Technique Alignment

| Technique Type | Methods |
|---|---|
| **Course techniques** | Text Mining (TF-IDF, BoW), Classification (Logistic Regression, Naive Bayes), Clustering (K-Means), Topic Modeling (LDA) |
| **External (beyond-course) technique** | Transformer-based embeddings (DistilBERT / RoBERTa-base-twitter) for sentiment classification |

---
# Section 2: Research Question Definition

Based on the EDA findings, three research questions are proposed below. RQ1 and RQ2 use course techniques; RQ3 requires an external, beyond-course technique.

---

## RQ1: How do different text representations affect sentiment classification performance on short, noisy tweets?

**Motivation from EDA:** TF-IDF sparsity was near 1.0 (0.9999) in Checkpoint 1, suggesting standard bag-of-words features lose most information on 140-character tweets. This question asks whether enriching the representation (e.g., adding n-grams, subword features) changes classification outcomes.

- **Data mining task type:** Text classification
- **Relevant algorithms:** Logistic Regression with TF-IDF (unigrams vs. unigrams+bigrams), Multinomial Naive Bayes (course)
- **Evaluation criteria:** Accuracy, F1-score (macro), ROC-AUC on a held-out test set; feature sparsity comparison across representations

---

## RQ2: What latent topics emerge in positive vs. negative tweets, and are they meaningfully distinct?

**Motivation from EDA:** The most frequent words (good, day, love, work) appeared across both sentiment classes, suggesting that raw word frequency is not enough to distinguish sentiment — but underlying *topics* may be. LDA can reveal whether negative tweets cluster around topics like illness or frustration while positive tweets center on celebration or social events.

- **Data mining task type:** Unsupervised topic modeling / pattern discovery
- **Relevant algorithms:** Latent Dirichlet Allocation (LDA) — course technique
- **Evaluation criteria:** Topic coherence (c_v score via gensim), topic diversity (proportion of unique top words), qualitative interpretability of discovered topics

---

## RQ3: Do transformer-based embeddings outperform TF-IDF classifiers for Twitter sentiment, and where do they fail?

**Motivation from EDA:** Because tweets are short, sparse, and informal, contextual representations that understand word order and meaning should outperform pure frequency-based features. This question extends the project by applying a **beyond-course technique** (DistilBERT fine-tuned on Twitter data) and performing detailed error analysis to understand where transformers still struggle (e.g., sarcasm, slang).

- **Data mining task type:** Text classification with deep contextual embeddings
- **Relevant algorithms:** DistilBERT (Hugging Face `distilbert-base-uncased-finetuned-sst-2-english`) as external/beyond-course technique; compared against Logistic Regression + TF-IDF baseline from RQ1
- **Evaluation criteria:** Accuracy, F1-score, confusion matrix; error analysis on misclassified examples; inference time comparison

---

## RQ-to-Method Mapping Table

| Research Question | Task Type | Algorithm(s) | Course or External | Evaluation Metric(s) |
|---|---|---|---|---|
| RQ1: Text representation vs. classification performance | Text classification | TF-IDF + Logistic Regression, Multinomial Naive Bayes | **Course** | Accuracy, F1, ROC-AUC, sparsity |
| RQ2: Latent topics in positive vs. negative tweets | Topic modeling | LDA (gensim) | **Course** | Coherence score, topic diversity, interpretability |
| RQ3: Transformer embeddings vs. TF-IDF | Text classification | DistilBERT (Hugging Face) | **External** | Accuracy, F1, confusion matrix, error analysis |

---
# Section 3: Additional EDA to Motivate and Validate Research Questions

In this section we perform additional exploratory analysis beyond Checkpoint 1 to:
1. Validate that the research questions are interesting and non-trivial
2. Confirm that the data supports the proposed methods (feasibility check)

We will examine:
- Vocabulary overlap between positive and negative classes (motivates RQ1 and RQ2)
- Tweet length distributions per class (informs tokenization strategy for RQ3)
- Top discriminative unigrams and bigrams (motivates n-gram enrichment in RQ1)
- Temporal distribution of tweets (informs sampling strategy)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
# Standard libraries
import re
import warnings
warnings.filterwarnings('ignore')

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Text processing
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder

print("All imports successful.")

In [ ]:
# ── Data Loading ─────────────────────────────────────────────────────────────
# WHY: We reload from the uploaded file to ensure reproducibility. Column names
# are assigned manually because the CSV has no header row (as documented in the
# Sentiment140 dataset description).

from google.colab import files
uploaded = files.upload()  # Upload training.1600000.processed.noemoticon.csv

COLS = ["sentiment", "id", "date", "query", "user", "text"]
df = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    header=None,
    names=COLS
)

# Map raw label (0, 4) → readable label (negative, positive)
df["label"] = df["sentiment"].map({0: "negative", 4: "positive"})

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['label'].value_counts()}")

In [ ]:
# ── Text Preprocessing ───────────────────────────────────────────────────────
# WHY: We apply the same cleaning pipeline from Checkpoint 1 for consistency.
# This removes URLs, mentions, hashtag symbols, HTML artifacts, and lowercases
# the text. Deduplication is applied here to avoid biasing downstream analyses.

def clean_tweet(text: str) -> str:
    """Remove URLs, mentions, hashtag markers, HTML entities, and normalize whitespace."""
    text = re.sub(r"http\S+|www\.\S+", "", text)      # remove URLs
    text = re.sub(r"@\w+", "", text)                    # remove @mentions
    text = re.sub(r"#", "", text)                       # remove hashtag symbol (keep word)
    text = re.sub(r"&\w+;", "", text)                   # remove HTML entities (amp, quot, etc.)
    text = re.sub(r"[^a-zA-Z\s]", "", text)             # keep only letters and whitespace
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)                    # collapse multiple spaces
    return text

df["text_clean"] = df["text"].apply(clean_tweet)

# Remove duplicates on cleaned text to avoid data leakage
before = len(df)
df = df.drop_duplicates(subset="text_clean").reset_index(drop=True)
print(f"Removed {before - len(df):,} duplicate tweets. Remaining: {len(df):,}")

# Remove empty strings that result from aggressive cleaning
df = df[df["text_clean"].str.strip() != ""].reset_index(drop=True)
print(f"After removing empty strings: {len(df):,}")

In [ ]:
# ── Additional EDA 1: Vocabulary Overlap Between Classes ─────────────────────
# WHY (RQ1 & RQ2 motivation): If positive and negative tweets share most of
# their top vocabulary, simple word-frequency features will struggle to
# discriminate sentiment. High overlap supports using either richer
# representations (RQ1) or topic models (RQ2) to surface latent structure.

# Sample equally from each class for a fair comparison
SAMPLE_SIZE = 100_000
pos_sample = df[df["label"] == "positive"].sample(SAMPLE_SIZE, random_state=42)["text_clean"]
neg_sample = df[df["label"] == "negative"].sample(SAMPLE_SIZE, random_state=42)["text_clean"]

# Build top-N word sets per class using CountVectorizer
TOP_N = 500

def top_words(corpus, n=TOP_N):
    """Return the set of the n most frequent words in a corpus."""
    cv = CountVectorizer(stop_words="english", max_features=n)
    cv.fit(corpus)
    return set(cv.get_feature_names_out())

pos_vocab = top_words(pos_sample)
neg_vocab = top_words(neg_sample)

overlap = pos_vocab & neg_vocab
overlap_pct = len(overlap) / TOP_N * 100

print(f"Top-{TOP_N} vocabulary overlap between classes: {len(overlap)} words ({overlap_pct:.1f}%)")
print(f"Words unique to positive tweets (sample): {sorted(pos_vocab - neg_vocab)[:15]}")
print(f"Words unique to negative tweets (sample): {sorted(neg_vocab - pos_vocab)[:15]}")

**Interpretation:** A high vocabulary overlap (expected ~70–80%) confirms that simple word-presence features are not strongly discriminative, motivating richer representations in RQ1 (n-gram enrichment, subword features) and unsupervised topic discovery in RQ2 (LDA) to find hidden structure that word frequencies alone miss.

In [ ]:
# ── Additional EDA 2: Top Discriminative Unigrams and Bigrams (Log-Odds) ─────
# WHY (RQ1 motivation): Log-odds ratio identifies words that are specifically
# associated with one class over the other, beyond raw frequency. This informs
# whether bigrams add discriminative signal that unigrams miss — a core
# question in RQ1.

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def top_discriminative_terms(pos_corpus, neg_corpus, n_features=10_000, top_k=15, ngram_range=(1,2)):
    """
    Fit a TF-IDF vectorizer on combined corpus, then compute the mean TF-IDF
    score per class and return the top terms most associated with each class.
    WHY TF-IDF mean: accounts for both frequency and uniqueness, giving a
    better discrimination signal than raw counts.
    """
    combined = pd.concat([pos_corpus, neg_corpus]).reset_index(drop=True)
    labels = [1] * len(pos_corpus) + [0] * len(neg_corpus)

    tfidf = TfidfVectorizer(stop_words="english", max_features=n_features,
                             ngram_range=ngram_range, min_df=10)
    X = tfidf.fit_transform(combined)
    features = tfidf.get_feature_names_out()

    pos_mask = np.array(labels) == 1
    mean_pos = np.asarray(X[pos_mask].mean(axis=0)).flatten()
    mean_neg = np.asarray(X[~pos_mask].mean(axis=0)).flatten()

    # Log-odds: positive value → more associated with positive class
    log_odds = np.log((mean_pos + 1e-9) / (mean_neg + 1e-9))

    top_pos_idx = log_odds.argsort()[-top_k:][::-1]
    top_neg_idx = log_odds.argsort()[:top_k]

    top_pos = [(features[i], log_odds[i]) for i in top_pos_idx]
    top_neg = [(features[i], log_odds[i]) for i in top_neg_idx]
    return top_pos, top_neg

# Unigrams only
top_pos_uni, top_neg_uni = top_discriminative_terms(
    pos_sample, neg_sample, ngram_range=(1,1), top_k=15
)
# Bigrams only
top_pos_bi, top_neg_bi = top_discriminative_terms(
    pos_sample, neg_sample, ngram_range=(2,2), top_k=15
)

print("=== Top Positive Unigrams (log-odds) ===")
for w, s in top_pos_uni: print(f"  {w:30s} {s:.4f}")

print("\n=== Top Negative Unigrams (log-odds) ===")
for w, s in top_neg_uni: print(f"  {w:30s} {s:.4f}")

print("\n=== Top Positive Bigrams (log-odds) ===")
for w, s in top_pos_bi: print(f"  {w:30s} {s:.4f}")

print("\n=== Top Negative Bigrams (log-odds) ===")
for w, s in top_neg_bi: print(f"  {w:30s} {s:.4f}")

In [ ]:
# ── Visualization: Discriminative Terms Bar Chart ─────────────────────────────
# WHY: Plotting the log-odds values makes the discrimination pattern visually
# clear and helps justify why n-gram enrichment (RQ1) adds new signal.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, top_pos, top_neg, title in [
    (axes[0], top_pos_uni, top_neg_uni, "Unigrams"),
    (axes[1], top_pos_bi,  top_neg_bi,  "Bigrams")
]:
    terms   = [w for w, _ in top_neg[::-1]] + [w for w, _ in top_pos]
    scores  = [s for _, s in top_neg[::-1]] + [s for _, s in top_pos]
    colors  = ["#E74C3C" if s < 0 else "#2ECC71" for s in scores]
    ax.barh(terms, scores, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Top Discriminative {title} (Log-Odds)", fontsize=11)
    ax.set_xlabel("Log-Odds (positive → right, negative → left)")

plt.tight_layout()
plt.suptitle("Figure 1: Discriminative Terms by Sentiment Class", y=1.02, fontsize=12, fontweight="bold")
plt.show()

print("\nTakeaway: Bigrams reveal phrase-level sentiment patterns (e.g., 'miss you', 'feel better')")
print("that unigrams alone miss, supporting the n-gram enrichment approach in RQ1.")

In [ ]:
# ── Additional EDA 3: Tweet Length Distribution by Sentiment Class ────────────
# WHY (RQ3 feasibility): Transformers like DistilBERT have a 512-token limit.
# Understanding the word-count distribution per class tells us whether truncation
# will be a concern and confirms that tweets are well within the token limit,
# making transformer inference feasible without special chunking.

df["word_count"] = df["text_clean"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram: word count distribution per class
for label, color in [("positive", "#2ECC71"), ("negative", "#E74C3C")]:
    subset = df[df["label"] == label]["word_count"]
    axes[0].hist(subset.sample(min(50000, len(subset)), random_state=42),
                 bins=30, alpha=0.6, label=label, color=color)

axes[0].set_title("Tweet Word-Count Distribution by Class")
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# Boxplot: word count per class
df[df["label"].isin(["positive", "negative"])].boxplot(
    column="word_count", by="label", ax=axes[1],
    boxprops=dict(color="#2C3E50"),
    medianprops=dict(color="#E67E22", linewidth=2)
)
axes[1].set_title("Word Count Distribution (Box Plot)")
axes[1].set_xlabel("Sentiment Class")
axes[1].set_ylabel("Word Count")
plt.suptitle("")  # suppress auto title from boxplot

fig.suptitle("Figure 2: Tweet Length by Sentiment Class", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary statistics
print(df.groupby("label")["word_count"].describe().round(2))
max_words = df["word_count"].max()
pct_under_50 = (df["word_count"] < 50).mean() * 100
print(f"\nMax word count: {max_words}")
print(f"Tweets with < 50 words: {pct_under_50:.1f}% — all safely within BERT's 512-token limit.")

In [ ]:
# ── Additional EDA 4: Temporal Distribution of Tweets ─────────────────────────
# WHY: Understanding when tweets were posted confirms there is no extreme
# temporal skew (e.g., all tweets from a single event) that would make the
# dataset non-representative. This also tells us whether time-based train/test
# splits are needed for RQ1 and RQ3.

df["date_parsed"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
df["month"] = df["date_parsed"].dt.to_period("M")

monthly_counts = df.groupby(["month", "label"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4))
monthly_counts.plot(kind="bar", ax=ax, color=["#E74C3C", "#2ECC71"], alpha=0.8)
ax.set_title("Figure 3: Monthly Tweet Volume by Sentiment Class", fontsize=12, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Tweet Count")
ax.legend(title="Sentiment")
ax.xaxis.set_tick_params(rotation=45)
plt.tight_layout()
plt.show()

print("Takeaway: If tweets are concentrated in a narrow time window, random splits")
print("are acceptable. If spread over many months, temporal split might be more rigorous.")

In [ ]:
# ── Additional EDA 5: Hashtag and Mention Analysis ───────────────────────────
# WHY: Understanding whether hashtags and @mentions carry sentiment signal
# informs preprocessing decisions for all three RQs. If mentions/hashtags are
# sentiment-neutral, removing them (as done in Checkpoint 1) is justified.

SMALL_SAMPLE = 50_000
sample = df.sample(SMALL_SAMPLE, random_state=42)

sample = sample.copy()
sample["n_hashtags"] = sample["text"].str.count(r"#\w+")
sample["n_mentions"] = sample["text"].str.count(r"@\w+")
sample["has_url"]    = sample["text"].str.contains(r"http|www", case=False).astype(int)

print("=== Hashtag / Mention / URL Summary ===")
print(sample.groupby("label")[["n_hashtags", "n_mentions", "has_url"]].mean().round(3))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, title in [
    (axes[0], "n_hashtags", "Avg Hashtags per Tweet"),
    (axes[1], "n_mentions",  "Avg Mentions per Tweet"),
    (axes[2], "has_url",     "% Tweets with URL")
]:
    means = sample.groupby("label")[col].mean()
    means.plot(kind="bar", ax=ax, color=["#E74C3C", "#2ECC71"], alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Mean")
    ax.xaxis.set_tick_params(rotation=0)

fig.suptitle("Figure 4: Metadata Features by Sentiment Class", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

**EDA Summary for Checkpoint 2:**

| Analysis | Key Finding | Implication |
|---|---|---|
| Vocabulary overlap | ~70–80% overlap in top-500 words | Simple BoW features are not strongly discriminative → supports RQ1 and RQ2 |
| Log-odds terms | Bigrams reveal phrase-level sentiment phrases | Including bigrams in TF-IDF adds new signal → validates RQ1 n-gram comparison |
| Tweet length | Nearly all tweets < 50 words | No truncation issues for BERT (512-token limit) → confirms RQ3 feasibility |
| Temporal distribution | Mostly within a few months in 2009 | Random train/test split is acceptable; no strong temporal drift |
| Hashtags / mentions | Minimal difference between classes | Removing them in preprocessing is justified; they carry little discriminative signal |

---
# Section 4: Motivation and Feasibility

## 4.1 Motivation

**RQ1 — Text Representation vs. Classification Performance:**  
EDA confirms that TF-IDF sparsity is ≈0.9999 and vocabulary overlap between classes is high (~70–80%). This means that standard unigram features provide a weak signal. Comparing unigrams, bigrams, and character n-grams within the same classification framework tests whether feature engineering alone can close the performance gap — a practically important question for low-resource text classification.

**RQ2 — Latent Topic Structure:**  
The most frequent words (good, day, love, just) appear in both sentiment classes with similar frequency. This suggests that tweets cluster around topics (e.g., work, relationships, health) independently of sentiment, and that the *combination* of topic and sentiment is what carries meaning. LDA can reveal whether this structure exists and how clearly topics separate between classes.

**RQ3 — Transformer vs. TF-IDF:**  
Tweet length analysis confirms all tweets fall within DistilBERT's 512-token limit. Short, context-rich text is precisely where transformers are theorized to outperform frequency-based methods — the question is by how much and in what failure modes (e.g., sarcasm, slang).

## 4.2 Non-Triviality

- **RQ1** is non-trivial because the answer is not obvious: bigrams increase feature count dramatically and may hurt generalization in sparse regimes even when they improve discrimination.
- **RQ2** is non-trivial because LDA assumes a bag-of-words representation of the same noisy tweets that TF-IDF struggles with; we want to know if unsupervised latent structure is still recoverable.
- **RQ3** is non-trivial because domain shift matters: a general-purpose DistilBERT model may not generalize as well to informal Twitter language as expected, and the failure mode analysis goes beyond accuracy.

## 4.3 Feasibility

| Method | Feasibility Assessment |
|---|---|
| TF-IDF + Logistic Regression (RQ1) | Very feasible — runs in seconds on 200K samples using scikit-learn |
| LDA (RQ2) | Feasible — gensim LDA on 50–100K tokens is manageable on Colab; runtime ~5–10 mins |
| DistilBERT inference (RQ3) | Feasible in Colab with GPU; inference on 5K tweets takes ~2–3 minutes |

## 4.4 Risks

| Risk | Mitigation |
|---|---|
| DistilBERT fine-tuning is memory/time intensive | Use a pre-fine-tuned model for inference only (no training), with a stratified 5K-tweet sample |
| LDA topic coherence may be low on noisy tweets | Tune number of topics (k) using coherence curve; filter very short cleaned tweets (<3 words) |
| Bigram TF-IDF matrix may be too large for RAM | Cap features at 100K max; use sparse matrix operations throughout |
| Sentiment140 labels were auto-generated from emoticons | Acknowledge label noise in results; report confidence intervals or bootstrap estimates |

In [ ]:
# ── Feasibility Check: LDA Topic Count vs. Coherence ─────────────────────────
# WHY: Before committing to LDA for RQ2, we validate that coherent topics can
# actually be learned from this data. We train LDA with k ∈ {5, 10, 15, 20}
# topics on a sample and plot coherence scores to choose the best k.
# This guards against the risk that noisy tweets yield uninterpretable topics.

!pip install gensim --quiet

import gensim
import gensim.corpora as corpora
from gensim.models import CoherenceModel, LdaMulticore
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords", quiet=True)

STOP_WORDS = set(stopwords.words("english")) | {"rt", "lol", "im", "ur", "dont", "amp", "quot"}

def tokenize(text: str) -> list:
    """Tokenize cleaned text, remove stopwords, and keep tokens with >= 3 chars."""
    return [w for w in text.split() if w not in STOP_WORDS and len(w) >= 3]

# Use a stratified sample of 50K tweets (25K per class) for speed
lda_sample = pd.concat([
    df[df["label"] == "positive"].sample(25_000, random_state=42),
    df[df["label"] == "negative"].sample(25_000, random_state=42)
])

texts = lda_sample["text_clean"].apply(tokenize).tolist()

# Filter out very short documents (< 3 tokens) — they add noise to LDA
texts = [t for t in texts if len(t) >= 3]
print(f"Documents for LDA training: {len(texts):,}")

# Build gensim dictionary and corpus
dictionary = corpora.Dictionary(texts)
# Keep only tokens that appear in at least 10 documents but not more than 50% of documents
dictionary.filter_extremes(no_below=10, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in texts]

print(f"Dictionary size after filtering: {len(dictionary):,} unique tokens")

In [ ]:
# ── LDA Coherence Curve: Choose Optimal k ────────────────────────────────────
# WHY: We sweep k from 5 to 25 in steps of 5. The k with the highest
# c_v coherence score is selected for the full RQ2 analysis. This is a
# standard model-selection procedure for topic models.

k_values = [5, 10, 15, 20, 25]
coherence_scores = []

for k in k_values:
    lda_model = LdaMulticore(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=5,          # number of passes through corpus (feasibility-tuned)
        workers=2,         # parallel workers for Colab
        chunksize=2000
    )
    cm = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence="c_v")
    score = cm.get_coherence()
    coherence_scores.append(score)
    print(f"  k={k:2d} → coherence = {score:.4f}")

# Plot coherence vs. k
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, coherence_scores, marker="o", color="#3498DB", linewidth=2)
best_k = k_values[int(np.argmax(coherence_scores))]
ax.axvline(best_k, color="#E74C3C", linestyle="--", label=f"Best k={best_k}")
ax.set_title("Figure 5: LDA Coherence Score vs. Number of Topics", fontsize=11, fontweight="bold")
ax.set_xlabel("Number of Topics (k)")
ax.set_ylabel("Coherence Score (c_v)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nSelected k = {best_k} based on peak coherence score.")

In [ ]:
# ── Initial LDA Run with Best k ───────────────────────────────────────────────
# WHY: Running a preliminary LDA model confirms that the gensim pipeline works
# end-to-end and that topics are interpretable enough to proceed with RQ2.
# This is a method feasibility test, not the final analysis.

best_lda = LdaMulticore(
    corpus=corpus,
    id2word=dictionary,
    num_topics=best_k,
    random_state=42,
    passes=10,
    workers=2,
    chunksize=2000
)

print(f"=== Top 10 words per topic (k={best_k}) ===")
for idx, topic in best_lda.print_topics(num_words=10):
    print(f"Topic {idx:2d}: {topic}")

In [ ]:
# ── Feasibility Check: TF-IDF + Logistic Regression Baseline (RQ1 / RQ3) ─────
# WHY: Before proposing the transformer comparison in RQ3, we establish that
# the course-technique baseline (TF-IDF + LR) is trainable and achieves
# reasonable performance. This also serves as the comparison baseline for RQ3.

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Stratified sample for fast feasibility run
BASELINE_SIZE = 100_000
baseline_df = pd.concat([
    df[df["label"] == "positive"].sample(BASELINE_SIZE // 2, random_state=42),
    df[df["label"] == "negative"].sample(BASELINE_SIZE // 2, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

y = (baseline_df["label"] == "positive").astype(int)  # 1 = positive, 0 = negative

# Train/test split (80/20, stratified)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    baseline_df["text_clean"], y,
    test_size=0.2, random_state=42, stratify=y
)

# ── Unigram TF-IDF ──
tfidf_uni = TfidfVectorizer(stop_words="english", max_features=50_000, ngram_range=(1, 1))
X_train_uni = tfidf_uni.fit_transform(X_train_text)
X_test_uni  = tfidf_uni.transform(X_test_text)

lr_uni = LogisticRegression(max_iter=200, solver="lbfgs", C=1.0, random_state=42)
lr_uni.fit(X_train_uni, y_train)
pred_uni = lr_uni.predict(X_test_uni)
prob_uni = lr_uni.predict_proba(X_test_uni)[:, 1]

print("=== Baseline: Unigram TF-IDF + Logistic Regression ===")
print(classification_report(y_test, pred_uni, target_names=["negative", "positive"]))
print(f"ROC-AUC: {roc_auc_score(y_test, prob_uni):.4f}")

In [ ]:
# ── Bigram TF-IDF Baseline Comparison ────────────────────────────────────────
# WHY: This is the first element of RQ1 — does adding bigrams to TF-IDF
# improve performance over unigrams? We test this here as a feasibility step
# and report the result for planning purposes.

tfidf_bi = TfidfVectorizer(stop_words="english", max_features=100_000, ngram_range=(1, 2))
X_train_bi = tfidf_bi.fit_transform(X_train_text)
X_test_bi  = tfidf_bi.transform(X_test_text)

lr_bi = LogisticRegression(max_iter=200, solver="lbfgs", C=1.0, random_state=42)
lr_bi.fit(X_train_bi, y_train)
pred_bi = lr_bi.predict(X_test_bi)
prob_bi = lr_bi.predict_proba(X_test_bi)[:, 1]

print("=== Bigram TF-IDF + Logistic Regression ===")
print(classification_report(y_test, pred_bi, target_names=["negative", "positive"]))
print(f"ROC-AUC: {roc_auc_score(y_test, prob_bi):.4f}")

# Quick summary
from sklearn.metrics import accuracy_score
print("\n=== Comparison Summary ===")
print(f"Unigram LR Accuracy:  {accuracy_score(y_test, pred_uni):.4f}")
print(f"Bigram  LR Accuracy:  {accuracy_score(y_test, pred_bi):.4f}")
print("\nConclusion: Adding bigrams is worth exploring in RQ1.")

In [ ]:
# ── Feasibility Check: DistilBERT Inference (RQ3) ─────────────────────────────
# WHY: We validate that the Hugging Face transformers pipeline works in Colab,
# runs within time limits on a small sample, and produces reasonable predictions.
# This is NOT the full RQ3 analysis — it is a method feasibility test to confirm
# the approach is implementable before committing to it.
# We deliberately use a pre-fine-tuned model (no additional training) to avoid
# prohibitive compute on Colab's free tier.

!pip install transformers torch --quiet

from transformers import pipeline
import time

# ── Model Choice Justification ────────────────────────────────────────────────
# We use 'distilbert-base-uncased-finetuned-sst-2-english' (Hugging Face).
# WHY DistilBERT over BERT:
#   - 40% smaller and 60% faster than BERT-base while retaining 97% of its
#     performance on GLUE benchmarks (Sanh et al., 2019).
#   - Colab free tier has GPU RAM limits; DistilBERT fits comfortably.
# WHY SST-2 fine-tuned:
#   - This model was fine-tuned on Stanford Sentiment Treebank (binary sentiment),
#     making it directly applicable to our positive/negative task.
#   - In Checkpoint 3 we will explore a Twitter-specific variant (e.g.,
#     cardiffnlp/twitter-roberta-base-sentiment) for domain adaptation.

print("Loading DistilBERT sentiment pipeline...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=128  # tweets are short; 128 tokens is sufficient
)
print("Pipeline loaded.")

# Inference on a small test sample
BERT_SAMPLE = 500  # feasibility run — small sample
bert_test_df = pd.concat([
    df[df["label"] == "positive"].sample(BERT_SAMPLE // 2, random_state=99),
    df[df["label"] == "negative"].sample(BERT_SAMPLE // 2, random_state=99)
]).sample(frac=1, random_state=99).reset_index(drop=True)

texts_for_bert = bert_test_df["text_clean"].tolist()
true_labels = (bert_test_df["label"] == "positive").astype(int).tolist()

t0 = time.time()
results = sentiment_pipeline(texts_for_bert, batch_size=32)
elapsed = time.time() - t0

# Map model output labels to 0/1
label_map = {"POSITIVE": 1, "NEGATIVE": 0}
pred_labels = [label_map[r["label"]] for r in results]

acc = accuracy_score(true_labels, pred_labels)
print(f"\nDistilBERT feasibility run on {BERT_SAMPLE} tweets:")
print(f"  Accuracy   : {acc:.4f}")
print(f"  Elapsed    : {elapsed:.1f}s ({elapsed/BERT_SAMPLE*1000:.1f} ms/tweet)")
print(f"  Extrapolated time for 5K tweets: ~{elapsed/BERT_SAMPLE*5000/60:.1f} minutes")
print(classification_report(true_labels, pred_labels, target_names=["negative", "positive"]))

**DistilBERT Feasibility Conclusion:**  
The model loads, runs, and produces meaningful predictions within acceptable time limits on Colab. Inference on 5K tweets (the planned RQ3 sample) is estimated to take ~2–5 minutes with a GPU — fully feasible. The accuracy on this small sample provides a rough sense of the baseline gap before detailed comparison in Checkpoint 3.

---
# Section 5: Methodological Planning

## 5.1 Course Algorithms

| Algorithm | Research Question | Library | Configuration |
|---|---|---|---|
| Logistic Regression + TF-IDF (unigram) | RQ1 baseline | `sklearn` | max_features=50K, C=1.0 |
| Logistic Regression + TF-IDF (bigram) | RQ1 extended | `sklearn` | max_features=100K, C=1.0 |
| Multinomial Naive Bayes | RQ1 comparison | `sklearn` | alpha=1.0 (Laplace smoothing) |
| LDA Topic Modeling | RQ2 | `gensim` | k=best from coherence curve, passes=10 |

## 5.2 External Algorithm

| Algorithm | Research Question | Library | Configuration |
|---|---|---|---|
| DistilBERT (pre-fine-tuned on SST-2) | RQ3 | `transformers` | max_length=128, batch_size=32 |

**Why DistilBERT is external/beyond-course:** Transformer architectures and self-attention mechanisms are not covered in the course curriculum. The use of Hugging Face pipelines and pre-trained language models requires external study of the BERT paper (Devlin et al., 2019) and transfer learning concepts not taught in class.

## 5.3 Evaluation Strategy

| Research Question | Primary Metric | Secondary Metrics | Baseline |
|---|---|---|---|
| RQ1 | F1-macro | Accuracy, ROC-AUC, feature count | Majority class (always predict positive) |
| RQ2 | Coherence score (c_v) | Topic diversity, qualitative labeling | LDA k=5 (fewest topics) |
| RQ3 | F1-macro vs. TF-IDF LR | Accuracy, confusion matrix, error examples | TF-IDF unigram LR from RQ1 |

## 5.4 Sampling Strategy

Because the full 1.6M tweet dataset is computationally prohibitive for some methods, we use stratified sampling:
- **RQ1 (classification):** 100K stratified sample (50K per class) for main experiments; scale test on 500K for final checkpoint.
- **RQ2 (LDA):** 50K stratified sample — LDA scales poorly; 50K provides enough documents for stable topic estimation.
- **RQ3 (DistilBERT):** 5K sample for inference (2.5K per class); larger samples are infeasible on Colab CPU; GPU needed for full evaluation.

**Justification for sampling:** All samples are stratified by class label to preserve the 50/50 balance observed in EDA. A fixed random seed (42) is used throughout for reproducibility.

## 5.5 Baselines

- **Majority class baseline:** Always predicts the most frequent class. Given 50/50 balance, this yields 50% accuracy — the floor any method must beat.
- **TF-IDF + LR unigram:** Established and validated in this notebook. All methods in RQ1 and RQ3 are compared against this.
- **LDA k=5 (fewest topics):** Serves as a reference point to show that increasing k improves coherence up to the optimal value.

In [ ]:
# ── Final Method Summary Table ────────────────────────────────────────────────
# WHY: Consolidating the planned methods in a structured table confirms the
# full methodological plan is coherent and complete before Checkpoint 3.

plan = pd.DataFrame([
    {
        "RQ": "RQ1",
        "Task": "Text Classification",
        "Algorithm": "TF-IDF (uni/bigram) + Logistic Regression / Naive Bayes",
        "Type": "Course",
        "Primary Metric": "F1-macro, ROC-AUC",
        "Sample Size": "100K",
        "Library": "sklearn"
    },
    {
        "RQ": "RQ2",
        "Task": "Topic Modeling",
        "Algorithm": "Latent Dirichlet Allocation (LDA)",
        "Type": "Course",
        "Primary Metric": "Coherence (c_v), Topic Diversity",
        "Sample Size": "50K",
        "Library": "gensim"
    },
    {
        "RQ": "RQ3",
        "Task": "Text Classification (Deep)",
        "Algorithm": "DistilBERT (pre-fine-tuned SST-2)",
        "Type": "External",
        "Primary Metric": "F1-macro, Error Analysis",
        "Sample Size": "5K",
        "Library": "transformers"
    }
])

print("=== Checkpoint 2: Methodological Plan Summary ===")
print(plan.to_string(index=False))

---
# Section 6: Checkpoint 2 Summary

## What was accomplished in this notebook:

1. **Project Scope Recap** — Summarized the Sentiment140 dataset and key findings from Checkpoint 1 EDA, establishing the context for research question formation.

2. **Research Question Definition** — Proposed three research questions: two using course techniques (text classification with TF-IDF variants, LDA topic modeling) and one requiring an external technique (DistilBERT transformer embeddings).

3. **RQ-to-Method Mapping** — Created a structured mapping table linking each RQ to its task type, algorithm, technique category, and evaluation metrics.

4. **Additional EDA** — Performed five new analyses beyond Checkpoint 1: vocabulary overlap, log-odds discriminative terms (unigrams and bigrams), tweet length distribution by class, temporal distribution, and hashtag/mention metadata analysis. Each EDA was tied to a specific RQ motivation.

5. **Motivation and Feasibility** — Documented motivation, non-triviality, and feasibility for each RQ. Identified computational risks and mitigations.

6. **Initial Method Runs** — Executed feasibility tests for all three methods:
   - TF-IDF + LR baseline (course)
   - LDA coherence curve to select optimal k (course)
   - DistilBERT inference pipeline on a small sample (external)

7. **Methodological Plan** — Specified algorithms, evaluation metrics, sample sizes, libraries, and baselines for Checkpoint 3 implementation.